# 🔍 BrandPulse AI — Classical NLP Baseline
### Notebook 01 · Preprocessing + Linear & Probabilistic Classifiers

---

**Goal**: Establish a robust, interpretable baseline for Twitter airline-sentiment classification using
TF-IDF vectorisation combined with three classical discriminative / generative models:

| # | Model | Type |
|---|---|---|
| 1 | Logistic Regression | Linear / Discriminative |
| 2 | Multinomial Naïve Bayes | Probabilistic / Generative |
| 3 | Linear SVC | Margin-based / Discriminative |

The best pipeline is serialised to `models/classical_tfidf_pipeline.joblib` for use in the Streamlit dashboard.

---
**Dataset**: [Twitter US Airline Sentiment](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment) · CrowdFlower / Figure Eight  
**Author** : BrandPulse AI — ML Engineering Team  
**Kernel**  : Python 3.10+ | scikit-learn 1.4+ | spaCy 3.7+ | NLTK 3.8+

---
## 0 · Environment Setup & Imports

In [1]:
# ── Standard library ──────────────────────────────────────────────
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# ── Numeric / Data ────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud

# ── Machine Learning ──────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
)
import joblib

# ── BrandPulse preprocessing module ──────────────────────────────
# Ensure the project root is on sys.path when running from notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import TextPreprocessor, preprocess_dataframe

# ── Global plotting style ─────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'      : 120,
    'figure.facecolor': '#0F1117',
    'axes.facecolor'  : '#0F1117',
    'axes.edgecolor'  : '#2A2D3E',
    'axes.labelcolor' : '#E0E0E0',
    'axes.titlecolor' : '#FFFFFF',
    'text.color'      : '#E0E0E0',
    'xtick.color'     : '#9E9E9E',
    'ytick.color'     : '#9E9E9E',
    'grid.color'      : '#2A2D3E',
    'grid.linewidth'  : 0.5,
    'font.family'     : 'sans-serif',
    'font.size'       : 11,
})
PALETTE = {'negative': '#FF4E6A', 'neutral': '#FFC947', 'positive': '#4ECDC4'}
LABEL_ORDER = ['negative', 'neutral', 'positive']

print('Environment ready.  Python', sys.version.split()[0])

Environment ready.  Python 3.13.7


---
## 1 · Data Ingestion

We load `Tweets.csv`, retain only the columns we need, and perform a quick sanity check on shapes and dtypes.

In [2]:
DATA_PATH  = os.path.join(PROJECT_ROOT, 'data', 'Tweets.csv')
MODEL_DIR  = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH, encoding='utf-8')
print(f'Loaded {len(df_raw):,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

Loaded 14,640 rows × 15 columns


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)


In [ ]:
# Keep only relevant columns and drop missing text rows
COLS_KEEP = ['airline_sentiment', 'text', 'airline']
df = df_raw[COLS_KEEP].copy()
df.dropna(subset=['text', 'airline_sentiment'], inplace=True)
df.rename(columns={'airline_sentiment': 'label'}, inplace=True)

print(f'Working dataset: {df.shape[0]:,} rows')
print('\nNull counts:')
print(df.isnull().sum())
df.dtypes

---
## 2 · Exploratory Data Analysis — Class & Airline Distributions

Understanding label imbalance is critical before choosing evaluation metrics and any class-weighting strategy.

In [ ]:
label_counts = df['label'].value_counts().reindex(LABEL_ORDER)
label_pct    = (label_counts / len(df) * 100).round(1)

print('=== Sentiment Class Distribution ===')
for lbl, cnt, pct in zip(LABEL_ORDER, label_counts, label_pct):
    bar = '█' * int(pct / 2)
    print(f'  {lbl:>10s} : {cnt:>5,}  ({pct:>5.1f}%)  {bar}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BrandPulse AI — Dataset Overview', fontsize=15, fontweight='bold', color='white', y=1.01)

# ── Left: Overall sentiment distribution ─────────────────────────
ax0 = axes[0]
colors = [PALETTE[l] for l in LABEL_ORDER]
wedges, texts, autotexts = ax0.pie(
    label_counts,
    labels=LABEL_ORDER,
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    wedgeprops=dict(edgecolor='#0F1117', linewidth=2),
    textprops={'color': 'white'},
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
ax0.set_title('Overall Sentiment Split', pad=12, fontsize=12)

# ── Right: Sentiment breakdown per airline ────────────────────────
ax1 = axes[1]
pivot = (
    df.groupby(['airline', 'label'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=LABEL_ORDER)
)
pivot.plot(
    kind='bar',
    ax=ax1,
    color=colors,
    edgecolor='#0F1117',
    linewidth=0.5,
    width=0.7,
)
ax1.set_title('Sentiment per Airline', fontsize=12)
ax1.set_xlabel('Airline')
ax1.set_ylabel('Tweet Count')
ax1.tick_params(axis='x', rotation=30)
ax1.legend(title='Sentiment', facecolor='#1E2030', edgecolor='#2A2D3E', labelcolor='white', title_fontsize=9)
ax1.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, '..', 'data', 'eda_distributions.png'), bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('Saved EDA distribution chart.')

---
## 3 · Text Preprocessing

We invoke the BrandPulse `TextPreprocessor` — a production-grade spaCy-primary / NLTK-fallback pipeline that:

- Strips URLs, `@handles`, and `#hashtags` (retaining keywords)
- Removes numbers and special characters
- Lemmatises tokens and removes English stop-words

> ⏱ Estimated runtime on ~14 k tweets: **30–90 s** (spaCy), **10–20 s** (NLTK fallback).

In [ ]:
tp = TextPreprocessor(spacy_model='en_core_web_sm', min_token_length=2)
print(f'NLP backend active: {tp.backend}')

In [ ]:
# Quick smoke-test before full run
samples = [
    '@AmericanAir your #service is TERRIBLE!! 😤 http://t.co/abc123',
    'I actually love @SouthwestAir — best flight ever! #happy',
    'Flight delayed by 3 hours with zero communication.',
    None,
    '   ',
]
print('=== Preprocessing Smoke Test ===')
for raw in samples:
    cleaned = tp.clean(raw)
    print(f'  RAW  : {str(raw)[:60]}')
    print(f'  CLEAN: {cleaned}')
    print()

In [ ]:
# Apply to full dataset
import time
t0 = time.time()
df = preprocess_dataframe(df, text_col='text', out_col='clean_text', preprocessor=tp)
elapsed = time.time() - t0

# Drop rows where cleaning produced an empty string
before = len(df)
df = df[df['clean_text'].str.strip().astype(bool)].reset_index(drop=True)
print(f'Preprocessing complete in {elapsed:.1f}s')
print(f'Dropped {before - len(df)} empty post-clean rows  →  {len(df):,} usable rows remain')
df[['text', 'clean_text', 'label']].sample(4, random_state=42)

---
## 4 · WordCloud Visualisations

WordClouds give an intuitive view of the most salient terms in positive vs. negative tweets after cleaning.

In [ ]:
def make_wordcloud(text_series: pd.Series, title: str, colormap: str, ax) -> None:
    """Render a WordCloud onto *ax* from a Series of cleaned tweet strings."""
    corpus = ' '.join(text_series.dropna().tolist())
    wc = WordCloud(
        width=900,
        height=450,
        background_color='#0F1117',
        colormap=colormap,
        max_words=120,
        collocations=False,
        prefer_horizontal=0.8,
        min_font_size=8,
    ).generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.axis('off')

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('BrandPulse AI — Most Frequent Terms by Sentiment', fontsize=14, fontweight='bold', color='white', y=1.01)

make_wordcloud(df.loc[df['label'] == 'negative', 'clean_text'],  'Negative Tweets',  'Reds',   axes[0])
make_wordcloud(df.loc[df['label'] == 'neutral',  'clean_text'],  'Neutral Tweets',   'YlOrBr', axes[1])
make_wordcloud(df.loc[df['label'] == 'positive', 'clean_text'],  'Positive Tweets',  'cool',   axes[2])

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, '..', 'data', 'wordclouds.png'), bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('Saved WordCloud visualisation.')

---
## 5 · Stratified Train / Test Split

An **80 / 20** stratified split ensures each fold mirrors the original class proportions, keeping evaluation honest under class imbalance.

In [ ]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f'Train : {len(X_train):,} samples')
print(f'Test  : {len(X_test):,} samples')
print()
print('Train label distribution (%):')
print((y_train.value_counts(normalize=True) * 100).round(1).to_string())
print()
print('Test  label distribution (%):')
print((y_test.value_counts(normalize=True) * 100).round(1).to_string())

---
## 6 · TF-IDF Vectorisation

We use **bigrams** (`ngram_range=(1,2)`) capped at **5,000 features**.  Bigrams capture common phrase patterns like *"great service"* or *"long delay"* that unigrams would miss.

Key hyperparameters:

| Parameter | Value | Rationale |
|---|---|---|
| `max_features` | 5 000 | Controls vocabulary size — balances expressiveness vs. curse-of-dimensionality |
| `ngram_range` | (1, 2) | Captures unigrams + bigrams |
| `sublinear_tf` | True | Applies log-normalisation to term frequencies, dampening highly repeated terms |
| `min_df` | 2 | Ignore terms appearing in fewer than 2 documents (typos / noise) |

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    strip_accents='unicode',
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(f'Vocabulary size : {len(tfidf.vocabulary_):,} features')
print(f'Train matrix    : {X_train_vec.shape}')
print(f'Test  matrix    : {X_test_vec.shape}')
print(f'Sparsity        : {1 - X_train_vec.nnz / (X_train_vec.shape[0] * X_train_vec.shape[1]):.4%}')

---
## 7 · Model Training & Evaluation

We define a unified `train_and_evaluate` helper that:
1. Fits the classifier on the TF-IDF training matrix.
2. Predicts on the held-out test set.
3. Prints a full **Classification Report** (Precision, Recall, F1, Accuracy).
4. Renders a **normalised Confusion Matrix** heatmap.

Results across all three classifiers are collected in `results_df` for side-by-side comparison.

In [ ]:
results = []  # Collect summary metrics


def train_and_evaluate(name: str, clf, X_tr, X_te, y_tr, y_te, label_order=LABEL_ORDER):
    """Fit *clf* on training data, evaluate on test data, and plot confusion matrix."""
    # ── Train ─────────────────────────────────────────────────────
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    # ── Classification report ─────────────────────────────────────
    report_dict = classification_report(y_te, y_pred, target_names=label_order, output_dict=True)
    report_str  = classification_report(y_te, y_pred, target_names=label_order)
    acc          = accuracy_score(y_te, y_pred)

    print(f'\n{"═" * 58}')
    print(f'  {name}')
    print(f'{"═" * 58}')
    print(report_str)

    # ── Normalised confusion matrix ────────────────────────────────
    cm = confusion_matrix(y_te, y_pred, labels=label_order, normalize='true')

    fig, ax = plt.subplots(figsize=(6, 5))
    fig.patch.set_facecolor('#0F1117')
    sns.heatmap(
        cm,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        xticklabels=label_order,
        yticklabels=label_order,
        linewidths=0.5,
        linecolor='#0F1117',
        vmin=0, vmax=1,
        ax=ax,
        cbar_kws={'shrink': 0.8},
    )
    ax.set_title(f'{name}\nNormalised Confusion Matrix', fontsize=11, fontweight='bold', color='white', pad=10)
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)
    ax.tick_params(axis='both', colors='#E0E0E0')
    plt.tight_layout()
    safe_name = name.lower().replace(' ', '_')
    plt.savefig(
        os.path.join(MODEL_DIR, '..', 'data', f'cm_{safe_name}.png'),
        bbox_inches='tight', facecolor='#0F1117'
    )
    plt.show()

    # ── Collect summary ───────────────────────────────────────────
    results.append({
        'Model'       : name,
        'Accuracy'    : round(acc, 4),
        'F1 (neg)'    : round(report_dict['negative']['f1-score'], 4),
        'F1 (neu)'    : round(report_dict['neutral']['f1-score'], 4),
        'F1 (pos)'    : round(report_dict['positive']['f1-score'], 4),
        'Macro F1'    : round(report_dict['macro avg']['f1-score'], 4),
        'Weighted F1' : round(report_dict['weighted avg']['f1-score'], 4),
        'clf_object'  : clf,  # kept for serialisation step
    })

    return clf

### 7a · Logistic Regression

A strong linear baseline.  We use `class_weight='balanced'` to compensate for the pronounced negative-class majority, and L2 regularisation with `C=1.0`.

In [ ]:
lr = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    random_state=42,
    n_jobs=-1,
)
train_and_evaluate('Logistic Regression', lr, X_train_vec, X_test_vec, y_train, y_test)

### 7b · Multinomial Naïve Bayes

A fast generative classifier that assumes conditional feature independence.  Works well with sparse TF-IDF matrices and serves as an interpretable probabilistic lower bound.

In [ ]:
mnb = MultinomialNB(alpha=0.1)  # Laplace smoothing (alpha=0.1 slightly less aggressive)
train_and_evaluate('Multinomial Naïve Bayes', mnb, X_train_vec, X_test_vec, y_train, y_test)

### 7c · Linear Support Vector Classifier

LinearSVC maximises the margin between classes in the high-dimensional TF-IDF space.  It is often the strongest linear model for text classification.

In [ ]:
svc = LinearSVC(
    C=0.5,
    max_iter=2000,
    class_weight='balanced',
    random_state=42,
)
train_and_evaluate('Linear SVC', svc, X_train_vec, X_test_vec, y_train, y_test)

---
## 8 · Results Comparison

A side-by-side table and grouped bar chart make it easy to identify the best-performing model overall and per class.

In [ ]:
metrics_df = pd.DataFrame(results).drop(columns='clf_object')
metrics_df = metrics_df.set_index('Model').sort_values('Weighted F1', ascending=False)

# Highlight the best value per column
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: #1D4E3F; color: #4ECDC4; font-weight: bold' if v else '' for v in is_max]

display(
    metrics_df.style
    .apply(highlight_max, axis=0)
    .format('{:.4f}')
    .set_caption('Model Comparison — Test-Set Metrics')
)

In [ ]:
# ── Grouped bar chart ─────────────────────────────────────────────
plot_cols  = ['Accuracy', 'Macro F1', 'Weighted F1', 'F1 (neg)', 'F1 (neu)', 'F1 (pos)']
plot_data  = metrics_df[plot_cols].T

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0F1117')

x      = np.arange(len(plot_cols))
n      = len(plot_data.columns)
width  = 0.22
bar_colors = ['#4ECDC4', '#FFC947', '#FF4E6A']

for i, (model_name, row) in enumerate(plot_data.items()):
    offset = (i - n / 2 + 0.5) * width
    bars = ax.bar(x + offset, row.values, width, label=model_name, color=bar_colors[i],
                  edgecolor='#0F1117', linewidth=0.4, alpha=0.9)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005, f'{h:.2f}',
                ha='center', va='bottom', fontsize=7, color='#CCCCCC')

ax.set_xticks(x)
ax.set_xticklabels(plot_cols, rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.08)
ax.set_title('Classical Classifier Comparison — TF-IDF (5k, bigrams)', fontsize=13, fontweight='bold', color='white')
ax.legend(facecolor='#1E2030', edgecolor='#2A2D3E', labelcolor='white')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, '..', 'data', 'model_comparison.png'), bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('Saved model comparison chart.')

---
## 9 · Persistence — Save Best Pipeline

We identify the best model by **Weighted F1**, wrap it with the fitted TF-IDF vectoriser into a scikit-learn `Pipeline`, and serialise to disk using `joblib`.

In [ ]:
# ── Identify best model ───────────────────────────────────────────
best_row    = max(results, key=lambda r: r['Weighted F1'])
best_name   = best_row['Model']
best_clf    = best_row['clf_object']

print(f'Best model  : {best_name}')
print(f'Weighted F1 : {best_row["Weighted F1"]:.4f}')
print(f'Accuracy    : {best_row["Accuracy"]:.4f}')

In [ ]:
# ── Rebuild Pipeline on FULL training data ────────────────────────
# We re-fit on train split (consistent with evaluation).
# For production you would retrain on 100% of available data.
final_pipeline = Pipeline([
    ('tfidf', tfidf),   # already fitted — transform only at predict time
    ('clf',   best_clf)
])

SAVE_PATH = os.path.join(MODEL_DIR, 'classical_tfidf_pipeline.joblib')
joblib.dump(final_pipeline, SAVE_PATH)
size_mb = os.path.getsize(SAVE_PATH) / 1024 / 1024

print(f'Pipeline saved → {SAVE_PATH}')
print(f'File size       : {size_mb:.2f} MB')

In [ ]:
# ── Sanity check: reload and predict ─────────────────────────────
loaded_pipeline = joblib.load(SAVE_PATH)

test_tweets = [
    '@Delta amazing flight, crew was wonderful!',
    '@united worst experience of my life, delayed 5 hours',
    'my bag arrived, nothing special about the flight',
]

# The pipeline handles raw text — TF-IDF transform is step 1
# We must pre-process first (pipeline only includes TF-IDF + clf, not the NLP cleaner)
cleaned_tweets = [tp.clean(t) for t in test_tweets]
predictions    = loaded_pipeline.predict(cleaned_tweets)

print('=== Live Prediction Test (reloaded pipeline) ===')
for tweet, pred in zip(test_tweets, predictions):
    emoji = {'positive': '😊', 'neutral': '😐', 'negative': '😡'}[pred]
    print(f'  {emoji} [{pred.upper():>8}]  {tweet[:70]}')

---
## 10 · Summary & Next Steps

| Model | Weighted F1 | Notes |
|---|---|---|
| Logistic Regression | — | Strong linear baseline, calibrated probabilities |
| Multinomial NB | — | Fastest to train, interpretable, but weaker on imbalanced data |
| **Linear SVC** | — | **Typically best** on sparse high-dim TF-IDF for text classification |

### Key Takeaways

1. **Class imbalance** (~63 % negative) depresses neutral/positive recall — `class_weight='balanced'` partially compensates.
2. **Bigrams** (`1,2`) capture phrase-level sentiment cues missed by unigrams alone.
3. **LinearSVC** is the recommended classical baseline for the Streamlit dashboard.

### What's Next — Notebook 02

> 🧠 `02_DeepLearning_LSTM.ipynb` — Build a sequence-aware **Bidirectional LSTM** using Keras/TensorFlow with an embedding layer, padding, and early stopping. LSTM models capture word-order context that bag-of-words approaches miss, and should further improve recall on the minority `positive` and `neutral` classes.

---
*BrandPulse AI | Classical NLP Baseline | Notebook 01*